# Lição 11 - Protocolo de Contexto do Modelo (MCP)

O **Model Context Protocol (MCP)** é um padrão aberto que permite que agentes descubram e utilizem dinamicamente ferramentas, recursos e fontes de dados em tempo de execução. Em vez de codificar ferramentas num agente, o MCP permite que os agentes se liguem a servidores externos que expõem capacidades sob demanda.

Nesta lição, irá aprender:
- O que é o MCP e por que é importante para sistemas de agentes
- Como funciona a arquitetura cliente-servidor do MCP
- Como construir agentes que utilizem descoberta de ferramentas no estilo MCP


## Configuração

**Pré-requisitos:**
- Projeto Azure AI Foundry com um modelo implantado
- Execute `az login` para autenticação


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## O que é o Protocolo de Contexto do Modelo (MCP)?

O MCP define uma forma padrão para que os agentes de IA descubram e interajam com ferramentas e fontes de dados externas:

- **Servidor MCP**: Expõe ferramentas, recursos e prompts através de um protocolo padrão
- **Cliente MCP**: O ambiente de execução do agente que se liga a servidores e descobre as capacidades disponíveis
- **Descoberta Dinâmica**: Os agentes não precisam de ferramentas codificadas — eles descobrem o que está disponível em tempo de execução

Isto é poderoso para construir sistemas de agentes extensíveis onde novas capacidades podem ser adicionadas sem modificar o código do agente.


## Como o MCP Funciona

```
┌─────────────┐     discover      ┌─────────────────┐
│  MCP Client  │ ──────────────► │   MCP Server     │
│  (Agent)     │                  │  (Tool Provider) │
│              │ ◄────────────── │                   │
│              │   tool results   │  • list_tools()  │
│              │                  │  • call_tool()   │
└─────────────┘                  │  • resources     │
                                  └─────────────────┘
```

1. O agente (MCP client) conecta-se a um MCP server
2. O MCP server responde com uma lista de ferramentas disponíveis e dos seus esquemas
3. O agente pode então invocar qualquer ferramenta descoberta durante o seu raciocínio
4. Os resultados são devolvidos através do mesmo protocolo


## Simulando a Descoberta de Ferramentas MCP

Como um servidor MCP real requer um processo de servidor em execução, demonstraremos o padrão usando funções `@tool` que simulam o que um serviço de alojamento ligado ao MCP forneceria.

Em produção, essas ferramentas seriam descobertas dinamicamente a partir de um servidor MCP em vez de serem definidas localmente.


In [ ]:
@tool(approval_mode="never_require")
def search_accommodations(
    location: Annotated[str, "The city to search for accommodations"],
    check_in: Annotated[str, "Check-in date (YYYY-MM-DD)"],
    check_out: Annotated[str, "Check-out date (YYYY-MM-DD)"],
    guests: Annotated[int, "Number of guests"] = 2
) -> str:
    """Search for accommodations (simulating an MCP-connected Airbnb tool).
    In production, this would be discovered via MCP from an accommodation service."""
    listings = {
        "Tokyo": [
            {"name": "Shinjuku Modern Apartment", "price": 120, "rating": 4.8},
            {"name": "Traditional Ryokan in Asakusa", "price": 200, "rating": 4.9},
            {"name": "Shibuya Studio", "price": 85, "rating": 4.5},
        ],
        "Paris": [
            {"name": "Le Marais Charming Flat", "price": 150, "rating": 4.7},
            {"name": "Montmartre Artist Loft", "price": 110, "rating": 4.6},
        ],
        "Barcelona": [
            {"name": "Gothic Quarter Penthouse", "price": 130, "rating": 4.8},
            {"name": "Barceloneta Beach Flat", "price": 95, "rating": 4.4},
        ],
    }
    results = listings.get(location, [])
    if not results:
        return f"No accommodations found in {location}"
    output = f"Accommodations in {location} ({check_in} to {check_out}, {guests} guests):\n"
    for listing in results:
        output += f"  - {listing['name']}: ${listing['price']}/night (★{listing['rating']})\n"
    return output


@tool(approval_mode="never_require")
def get_local_experiences(
    location: Annotated[str, "The city to find experiences in"],
    interest: Annotated[str, "Type of experience (food, culture, adventure, etc.)"] = "all"
) -> str:
    """Get local experiences and activities (simulating an MCP-connected tourism tool)."""
    experiences = {
        "Tokyo": {
            "food": ["Tsukiji Market Tour ($45)", "Ramen Making Class ($60)", "Sake Tasting ($35)"],
            "culture": ["Tea Ceremony ($50)", "Samurai Museum ($15)", "Sumo Tournament ($80)"],
            "adventure": ["Mt. Fuji Day Trip ($120)", "Go-kart City Tour ($80)"],
        },
        "Paris": {
            "food": ["Wine & Cheese Tasting ($55)", "Cooking Class ($90)", "Market Tour ($40)"],
            "culture": ["Louvre Guided Tour ($35)", "Montmartre Art Walk ($25)"],
        },
    }
    city_exp = experiences.get(location, {})
    if not city_exp:
        return f"No experiences found in {location}"
    if interest != "all" and interest in city_exp:
        items = city_exp[interest]
        return f"{interest.title()} experiences in {location}:\n" + "\n".join(f"  - {e}" for e in items)
    output = f"All experiences in {location}:\n"
    for cat, items in city_exp.items():
        output += f"\n  {cat.title()}:\n"
        for item in items:
            output += f"    - {item}\n"
    return output

## Construindo um Agente com Ferramentas ao Estilo MCP


In [ ]:
agent = await provider.create_agent(
    tools=[search_accommodations, get_local_experiences],
    name="AccommodationAgent",
    instructions="""You are an accommodation and travel experiences specialist powered by MCP-connected services.

Help travelers find the perfect place to stay and things to do. When searching:
1. Use the search_accommodations tool to find listings
2. Use the get_local_experiences tool to suggest activities
3. Compare options and make personalized recommendations
4. Consider the traveler's budget, interests, and travel style""",
)

response = await agent.run(
    "I'm visiting Tokyo for 5 nights in April with my partner. We love traditional Japanese culture and food. "
    "Find us a place to stay and suggest some experiences.",
    )
print(response)

## MCP em Produção

Num ambiente de produção, o MCP permite padrões poderosos:

- **Descoberta dinâmica de ferramentas**: Os agentes conectam-se a servidores MCP e descobrem ferramentas em tempo de execução
- **Arquitetura desacoplada**: Os fornecedores de ferramentas podem atualizar-se de forma independente do agente
- **Partilha entre organizações**: As equipas podem expor capacidades através de servidores MCP que qualquer agente pode usar
- **Suporte do Microsoft Agent Framework**: O MAF tem suporte incorporado ao cliente MCP através da integração `mcp`

Para usar um servidor MCP real com o MAF, ligaria via `hosted_mcp_tool()` ou através da integração do cliente MCP.

**Saiba mais:**
- [Especificação MCP](https://modelcontextprotocol.io/)
- [Suporte MCP do Microsoft Agent Framework](https://github.com/microsoft/agent-framework/tree/main/python/samples/02-agents/mcp)


## Resumo

- **MCP** é um padrão aberto para a descoberta dinâmica de ferramentas entre agentes e fornecedores de ferramentas
- A **arquitetura cliente-servidor** permite que os agentes descubram capacidades em tempo de execução
- MCP permite **sistemas de agentes extensíveis e desacoplados** onde ferramentas podem ser adicionadas sem alterações de código
- Microsoft Agent Framework fornece **suporte MCP integrado** para utilização em produção


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
Isenção de responsabilidade:
Este documento foi traduzido utilizando o serviço de tradução automática por IA Co‑op Translator (https://github.com/Azure/co-op-translator). Embora nos esforcemos pela precisão, por favor, tenha em atenção que traduções automáticas podem conter erros ou imprecisões. O documento original, na sua língua nativa, deve ser considerado a fonte autoritativa. Para informação crítica, recomenda-se uma tradução profissional efetuada por um tradutor humano. Não nos responsabilizamos por quaisquer mal-entendidos ou interpretações erradas decorrentes do uso desta tradução.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
